# AutoPilot — Wan 2.2 TI2V 5B Headless API
Runs the official **Wan 2.2 TI2V 5B** (Text/Image-to-Video) model via a headless ComfyUI FastAPI server exposed through Ngrok.

Your local AutoPilot pipeline sends `POST /generate_video` and gets back an MP4.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers
!pip install -q diffusers transformers accelerate safetensors huggingface_hub
!pip install -q opencv-python pillow numpy requests imageio imageio-ffmpeg
!pip install -q fastapi uvicorn pyngrok python-multipart
!apt-get -y install -qq aria2 ffmpeg
print("✅ Dependencies installed")


In [ ]:
# Cell 2 — Clone ComfyUI
import os, sys

COMFY_DIR = '/kaggle/working/ComfyUI'
if not os.path.exists(COMFY_DIR):
    os.system('cd /kaggle/working && git clone https://github.com/comfyanonymous/ComfyUI.git')
    print('✅ ComfyUI cloned')
else:
    print('✅ ComfyUI already present')

sys.path.insert(0, COMFY_DIR)
print("✅ ComfyUI on sys.path")


In [ ]:
# Cell 3 — Download Wan 2.2 model files (runs once; cached on re-run)
import os

BASE = 'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files'
WAN21_BASE = 'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files'

DIRS = {
    'diffusion': '/kaggle/working/ComfyUI/models/diffusion_models',
    'text_enc' : '/kaggle/working/ComfyUI/models/text_encoders',
    'vae'      : '/kaggle/working/ComfyUI/models/vae',
    'clip_vis' : '/kaggle/working/ComfyUI/models/clip_vision',
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, fname):
    dest = os.path.join(dest_dir, fname)
    if os.path.exists(dest):
        print(f'  ⏭  Already cached: {fname}')
        return
    print(f'  📥 Downloading {fname} ...')
    os.system(f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M '{url}' -d '{dest_dir}' -o '{fname}'")
    print(f'  ✅ {fname} done')

print('Downloading Wan 2.2 TI2V 5B diffusion model (~9.3 GB)...')
dl(f'{BASE}/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors',
   DIRS['diffusion'], 'wan2.2_ti2v_5B_fp16.safetensors')

print('Downloading UMT5-XXL text encoder FP8 (~6.3 GB)...')
dl(f'{BASE}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
   DIRS['text_enc'], 'umt5_xxl_fp8_e4m3fn_scaled.safetensors')

print('Downloading Wan 2.2 VAE (~1.3 GB)...')
dl(f'{BASE}/vae/wan2.2_vae.safetensors',
   DIRS['vae'], 'wan2.2_vae.safetensors')

print('Downloading CLIP Vision H (~0.6 GB)...')
dl(f'{WAN21_BASE}/clip_vision/clip_vision_h.safetensors',
   DIRS['clip_vis'], 'clip_vision_h.safetensors')

print("\n✅ All models downloaded")


In [ ]:
# Cell 4 — Verify GPU
import torch

print('GPU Information:')
print('=' * 60)
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB')
    print('\n✅ Ready to serve')
else:
    raise RuntimeError('No GPU — go to Kaggle Settings > Accelerator and select GPU T4 x2')


In [ ]:
# Cell 5 — Headless ComfyUI + FastAPI + Ngrok
#
# BEFORE RUNNING:
#   Replace YOUR_NGROK_TOKEN with a free token from https://dashboard.ngrok.com/get-started/your-authtoken
#
# AFTER RUNNING:
#   Copy the printed URL and add to your local .env:
#   KAGGLE_NGROK_URL=https://xxxx.ngrok-free.app

NGROK_TOKEN = 'YOUR_NGROK_TOKEN'  # <-- REPLACE THIS

# ─────────────────────────────────────────────────────────────────────────
import gc, os, random, sys
import numpy as np
from pathlib import Path
import torch
import imageio

sys.path.insert(0, '/kaggle/working/ComfyUI')
from nodes import NODE_CLASS_MAPPINGS

OUTPUT_DIR = '/kaggle/working/output'
INPUT_DIR  = '/kaggle/working/input'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(INPUT_DIR,  exist_ok=True)

def _clear():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def generate_video(
    image_path,
    positive_prompt,
    negative_prompt='text, watermark, ugly, worst quality, low quality, blurry',
    width=832,
    height=480,
    seed=0,
    steps=30,
    cfg=6.0,
    frames=49,
    fps=16,
):
    """Run Wan 2.2 TI2V 5B via headless ComfyUI nodes. Returns path to MP4."""
    if seed == 0:
        seed = random.randint(0, 2**32 - 1)

    with torch.inference_mode():
        # 1. Text encoder
        print('[1/6] Loading text encoder...')
        clip    = NODE_CLASS_MAPPINGS['CLIPLoader']().load_clip(
                      'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'wan', 'default')[0]
        pos_cond = NODE_CLASS_MAPPINGS['CLIPTextEncode']().encode(clip, positive_prompt)[0]
        neg_cond = NODE_CLASS_MAPPINGS['CLIPTextEncode']().encode(clip, negative_prompt)[0]
        del clip; _clear()

        # 2. CLIP Vision + image (I2V) or skip (T2V)
        loaded_image = None
        clip_vis_out = None
        if image_path and Path(image_path).exists():
            print('[2/6] Encoding reference image (I2V mode)...')
            loaded_image = NODE_CLASS_MAPPINGS['LoadImage']().load_image(image_path)[0]
            clip_vis     = NODE_CLASS_MAPPINGS['CLIPVisionLoader']().load_clip('clip_vision_h.safetensors')[0]
            clip_vis_out = NODE_CLASS_MAPPINGS['CLIPVisionEncode']().encode(clip_vis, loaded_image, 'none')[0]
            del clip_vis; _clear()
        else:
            print('[2/6] No image — running in T2V mode')

        # 3. VAE
        print('[3/6] Loading VAE...')
        vae = NODE_CLASS_MAPPINGS['VAELoader']().load_vae('wan2.2_vae.safetensors')[0]

        # 4. Prepare latents (TI2V conditioning)
        print('[4/6] Preparing latents...')
        if loaded_image is not None:
            # Try WanImageToVideo node (ComfyUI >= 0.3.x ships this for Wan 2.2)
            wan_node_cls = NODE_CLASS_MAPPINGS.get('WanImageToVideo')
            if wan_node_cls is None:
                # Fallback key name
                wan_node_cls = NODE_CLASS_MAPPINGS.get('Wan Image To Video')
            if wan_node_cls:
                pos_cond, neg_cond, latents = wan_node_cls().encode(
                    pos_cond, neg_cond, vae,
                    width, height, frames, 1,
                    loaded_image, clip_vis_out
                )
            else:
                # Older ComfyUI — use empty latent and rely on text prompt only
                print('  WanImageToVideo node not found, falling back to T2V')
                latents = NODE_CLASS_MAPPINGS['EmptyLatentImage']().generate(width, height, 1)[0]
        else:
            latents = NODE_CLASS_MAPPINGS['EmptyLatentImage']().generate(width, height, 1)[0]

        # 5. UNet + KSampler
        print('[5/6] Loading UNet and sampling (this takes a few minutes)...')
        model = NODE_CLASS_MAPPINGS['UNETLoader']().load_unet(
                    'wan2.2_ti2v_5B_fp16.safetensors', 'default')[0]
        sampled = NODE_CLASS_MAPPINGS['KSampler']().sample(
            model=model, seed=seed, steps=steps, cfg=cfg,
            sampler_name='euler', scheduler='simple',
            positive=pos_cond, negative=neg_cond,
            latent_image=latents, denoise=1.0
        )[0]
        del model; _clear()

        # 6. VAE decode + save MP4
        print('[6/6] Decoding and saving...')
        decoded  = NODE_CLASS_MAPPINGS['VAEDecode']().decode(vae, sampled)[0]
        del vae, sampled; _clear()

        out_path = f'{OUTPUT_DIR}/wan22_{seed}.mp4'
        frames_np = [(f.cpu().numpy() * 255).astype(np.uint8) for f in decoded]
        with imageio.get_writer(out_path, fps=fps) as writer:
            for frame in frames_np:
                writer.append_data(frame)

        print(f'✅ Video saved: {out_path}')
        return out_path


# ── FastAPI ──────────────────────────────────────────────────────────────────
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
import uvicorn
from pyngrok import ngrok

app = FastAPI(title='AutoPilot Wan2.2 API', version='1.0.0')

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'wan2.2_ti2v_5B_fp16'}


@app.post('/generate_video')
async def api_generate(
    image:  UploadFile = File(None),
    prompt: str = Form(...),
    seed:   int = Form(0),
    steps:  int = Form(30),
    width:  int = Form(832),
    height: int = Form(480),
    frames: int = Form(49),
):
    try:
        img_path = None
        if image is not None and image.filename:
            img_path = f'{INPUT_DIR}/{image.filename}'
            with open(img_path, 'wb') as fh:
                fh.write(await image.read())

        out_mp4 = generate_video(
            image_path=img_path,
            positive_prompt=prompt,
            seed=seed, steps=steps,
            width=width, height=height, frames=frames,
        )
        return FileResponse(out_mp4, media_type='video/mp4', filename='output.mp4')

    except Exception as exc:
        import traceback
        raise HTTPException(status_code=500, detail=traceback.format_exc())


# ── Launch ───────────────────────────────────────────────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)
print('\n' + '=' * 60)
print('  AUTOPILOT WAN 2.2 API READY')
print(f'  Public URL : {tunnel.public_url}')
print(f'  Add to .env: KAGGLE_NGROK_URL={tunnel.public_url}')
print(f'  Health check: {tunnel.public_url}/health')
print('=' * 60 + '\n')
uvicorn.run(app, host='0.0.0.0', port=8000)
